In [ ]:

import os

print("Root Directory Files:")
print(os.listdir("/"))

print("\nContent Directory Files:")
print(os.listdir("/content"))


In [ ]:

# Step 3.3 - Load Cleaned Master Sales Dataset

import pandas as pd

# Load cleaned dataset
df = pd.read_csv(
    "/content/master_sales_cleaned.csv",
    parse_dates=["engage_date", "close_date"]
)

# Display dataset dimensions
print("Dataset Shape:", df.shape)

print("\nTotal Records:", len(df))

print("\nTotal Columns:", len(df.columns))

print("\nFirst 5 Records:")
display(df.head())


In [ ]:

# Step 3.4 - Overall Sales KPI Analysis

# Total sales opportunities
total_deals = len(df)

# Won and lost deals
won_deals = (df["deal_stage"] == "Won").sum()
lost_deals = (df["deal_stage"] == "Lost").sum()

# Active opportunities
active_deals = df["deal_stage"].isin(
    ["Prospecting", "Engaging"]
).sum()

# Total won revenue
total_revenue = df.loc[
    df["deal_stage"] == "Won",
    "close_value"
].sum()

# Conversion rate based on completed deals
completed_deals = won_deals + lost_deals

conversion_rate = (
    won_deals / completed_deals * 100
    if completed_deals > 0 else 0
)

# Average won deal value
avg_deal_value = (
    total_revenue / won_deals
    if won_deals > 0 else 0
)

print("SALESPULSE AI - SALES KPI SUMMARY")
print("=" * 45)

print(f"Total Opportunities: {total_deals:,}")
print(f"Won Deals: {won_deals:,}")
print(f"Lost Deals: {lost_deals:,}")
print(f"Active Deals: {active_deals:,}")
print(f"Total Won Revenue: {total_revenue:,.2f}")
print(f"Conversion Rate: {conversion_rate:.2f}%")
print(f"Average Won Deal Value: {avg_deal_value:,.2f}")


In [ ]:

# Step 3.5 - Monthly Sales Revenue Trend

import matplotlib.pyplot as plt

# Filter successfully won deals
won_sales = df[df["deal_stage"] == "Won"].copy()

# Extract month from close date
won_sales["sales_month"] = (
    won_sales["close_date"].dt.to_period("M")
)

# Calculate monthly revenue
monthly_revenue = (
    won_sales.groupby("sales_month")["close_value"]
    .sum()
    .sort_index()
)

# Display monthly revenue
print("MONTHLY SALES REVENUE")
print("=" * 40)
print(monthly_revenue)

# Plot monthly revenue trend
plt.figure(figsize=(12, 5))

plt.plot(
    monthly_revenue.index.astype(str),
    monthly_revenue.values,
    marker="o"
)

plt.title("SalesPulse AI - Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:

# Step 3.6 - Deal Stage Distribution Analysis

import matplotlib.pyplot as plt

# Count deals by stage
deal_stage_counts = df["deal_stage"].value_counts()

# Calculate percentages
deal_stage_percentages = (
    df["deal_stage"].value_counts(normalize=True) * 100
)

# Display results
print("DEAL STAGE DISTRIBUTION")
print("=" * 40)

for stage in deal_stage_counts.index:
    print(
        f"{stage}: {deal_stage_counts[stage]:,} "
        f"({deal_stage_percentages[stage]:.2f}%)"
    )

# Create bar chart
plt.figure(figsize=(9, 5))

plt.bar(
    deal_stage_counts.index,
    deal_stage_counts.values,
    color="steelblue"
)

plt.title("SalesPulse AI - Deal Stage Distribution")
plt.xlabel("Deal Stage")
plt.ylabel("Number of Opportunities")

plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:

# Step 3.7 - Top-Performing Products by Revenue

import matplotlib.pyplot as plt

# Filter won deals
won_sales = df[df["deal_stage"] == "Won"].copy()

# Calculate total revenue by product
product_revenue = (
    won_sales.groupby("product")["close_value"]
    .sum()
    .sort_values(ascending=False)
)

# Display product revenue
print("PRODUCT REVENUE ANALYSIS")
print("=" * 45)
print(product_revenue)

# Create bar chart
plt.figure(figsize=(10, 5))

plt.bar(
    product_revenue.index,
    product_revenue.values,
    color="steelblue"
)

plt.title("SalesPulse AI - Revenue by Product")
plt.xlabel("Product")
plt.ylabel("Total Revenue")

plt.xticks(rotation=45)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:

# Step 3.8 - Sales Agent Performance Analysis

import matplotlib.pyplot as plt

# Filter won deals
won_sales = df[df["deal_stage"] == "Won"].copy()

# Calculate revenue by sales agent
agent_revenue = (
    won_sales.groupby("sales_agent")["close_value"]
    .sum()
    .sort_values(ascending=False)
)

# Display top 10 sales agents
top_10_agents = agent_revenue.head(10)

print("TOP 10 SALES AGENTS BY REVENUE")
print("=" * 45)
print(top_10_agents)

# Create horizontal bar chart
plt.figure(figsize=(10, 6))

plt.barh(
    top_10_agents.index,
    top_10_agents.values,
    color="steelblue"
)

plt.title("SalesPulse AI - Top 10 Sales Agents by Revenue")
plt.xlabel("Total Won Revenue")
plt.ylabel("Sales Agent")

plt.gca().invert_yaxis()
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:

# Step 3.9 - Regional Sales Performance Analysis

import matplotlib.pyplot as plt

# Filter successfully won deals
won_sales = df[df["deal_stage"] == "Won"].copy()

# Calculate regional sales performance
regional_performance = (
    won_sales.groupby("regional_office")
    .agg(
        total_revenue=("close_value", "sum"),
        won_deals=("opportunity_id", "count")
    )
    .sort_values("total_revenue", ascending=False)
)

# Display results
print("REGIONAL SALES PERFORMANCE")
print("=" * 45)
print(regional_performance)

# Create bar chart
plt.figure(figsize=(9, 5))

plt.bar(
    regional_performance.index,
    regional_performance["total_revenue"],
    color="steelblue"
)

plt.title("SalesPulse AI - Revenue by Region")
plt.xlabel("Regional Office")
plt.ylabel("Total Won Revenue")

plt.xticks(rotation=30)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:

# Step 3.10 - Industry-Wise Sales Performance Analysis

import matplotlib.pyplot as plt

# Filter won deals
won_sales = df[df["deal_stage"] == "Won"].copy()

# Replace missing industry names
won_sales["sector"] = won_sales["sector"].fillna(
    "Unknown Sector"
)

# Calculate industry performance
industry_performance = (
    won_sales.groupby("sector")
    .agg(
        total_revenue=("close_value", "sum"),
        won_deals=("opportunity_id", "count")
    )
    .sort_values("total_revenue", ascending=False)
)

# Display results
print("INDUSTRY-WISE SALES PERFORMANCE")
print("=" * 45)
print(industry_performance)

# Create horizontal bar chart
plt.figure(figsize=(11, 6))

plt.barh(
    industry_performance.index,
    industry_performance["total_revenue"],
    color="steelblue"
)

plt.title("SalesPulse AI - Revenue by Industry")
plt.xlabel("Total Won Revenue")
plt.ylabel("Industry")

plt.gca().invert_yaxis()
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:

# Step 3.10 - Industry-Wise Sales Performance Analysis

import matplotlib.pyplot as plt

# Filter won deals
won_sales = df[df["deal_stage"] == "Won"].copy()

# Replace missing industry names
won_sales["sector"] = won_sales["sector"].fillna(
    "Unknown Sector"
)

# Calculate industry performance
industry_performance = (
    won_sales.groupby("sector")
    .agg(
        total_revenue=("close_value", "sum"),
        won_deals=("opportunity_id", "count")
    )
    .sort_values("total_revenue", ascending=False)
)

# Display results
print("INDUSTRY-WISE SALES PERFORMANCE")
print("=" * 45)
print(industry_performance)

# Create horizontal bar chart
plt.figure(figsize=(11, 6))

plt.barh(
    industry_performance.index,
    industry_performance["total_revenue"],
    color="steelblue"
)

plt.title("SalesPulse AI - Revenue by Industry")
plt.xlabel("Total Won Revenue")
plt.ylabel("Industry")

plt.gca().invert_yaxis()
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:

# Step 3.11 - Sales Conversion Rate by Region

import matplotlib.pyplot as plt

# Filter completed deals
completed_sales = df[
    df["deal_stage"].isin(["Won", "Lost"])
].copy()

# Calculate regional performance
regional_conversion = (
    completed_sales.groupby("regional_office")
    .agg(
        total_completed=("opportunity_id", "count"),
        won_deals=("deal_stage", lambda x: (x == "Won").sum())
    )
)

# Calculate conversion rate
regional_conversion["conversion_rate"] = (
    regional_conversion["won_deals"]
    / regional_conversion["total_completed"]
    * 100
)

# Sort by conversion rate
regional_conversion = regional_conversion.sort_values(
    "conversion_rate",
    ascending=False
)

# Display results
print("REGIONAL SALES CONVERSION ANALYSIS")
print("=" * 50)
print(regional_conversion.round(2))

# Create bar chart
plt.figure(figsize=(9, 5))

plt.bar(
    regional_conversion.index,
    regional_conversion["conversion_rate"],
    color="steelblue"
)

plt.title("SalesPulse AI - Conversion Rate by Region")
plt.xlabel("Regional Office")
plt.ylabel("Conversion Rate (%)")
plt.ylim(0, 100)

plt.xticks(rotation=30)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:

# Step 3.12 - Sales Cycle Duration Analysis

import matplotlib.pyplot as plt

# Filter won deals
won_sales = df[df["deal_stage"] == "Won"].copy()

# Calculate sales cycle duration in days
won_sales["sales_cycle_days"] = (
    won_sales["close_date"] - won_sales["engage_date"]
).dt.days

# Remove invalid or missing durations
won_sales = won_sales[
    won_sales["sales_cycle_days"].notna()
    & (won_sales["sales_cycle_days"] >= 0)
]

# Calculate key metrics
avg_cycle = won_sales["sales_cycle_days"].mean()
median_cycle = won_sales["sales_cycle_days"].median()
min_cycle = won_sales["sales_cycle_days"].min()
max_cycle = won_sales["sales_cycle_days"].max()

print("SALES CYCLE DURATION ANALYSIS")
print("=" * 45)

print(f"Average Sales Cycle: {avg_cycle:.2f} days")
print(f"Median Sales Cycle: {median_cycle:.2f} days")
print(f"Minimum Sales Cycle: {min_cycle:.0f} days")
print(f"Maximum Sales Cycle: {max_cycle:.0f} days")
print(f"Valid Won Deals Analyzed: {len(won_sales):,}")

# Create histogram
plt.figure(figsize=(10, 5))

plt.hist(
    won_sales["sales_cycle_days"],
    bins=20,
    color="steelblue",
    edgecolor="black"
)

plt.title("SalesPulse AI - Sales Cycle Duration")
plt.xlabel("Sales Cycle Duration (Days)")
plt.ylabel("Number of Won Deals")

plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:

# Step 3.13 - Sales Performance Insights

print("SALESPULSE AI - BUSINESS INSIGHTS")
print("=" * 55)

# Insight 1: Overall revenue
print(f"\n1. Total Won Revenue: {total_revenue:,.2f}")

# Insight 2: Conversion rate
print(f"2. Sales Conversion Rate: {conversion_rate:.2f}%")

# Insight 3: Best-performing product
top_product = product_revenue.idxmax()
print(f"3. Highest Revenue Product: {top_product}")

# Insight 4: Best-performing sales agent
top_agent = agent_revenue.idxmax()
print(f"4. Top Sales Agent: {top_agent}")

# Insight 5: Highest revenue region
top_region = regional_performance["total_revenue"].idxmax()
print(f"5. Highest Revenue Region: {top_region}")

# Insight 6: Highest revenue industry
top_industry = industry_performance["total_revenue"].idxmax()
print(f"6. Highest Revenue Industry: {top_industry}")

# Insight 7: Best conversion region
best_conversion_region = regional_conversion[
    "conversion_rate"
].idxmax()

print(
    f"7. Best Conversion Region: "
    f"{best_conversion_region}"
)

# Insight 8: Average sales cycle
print(f"8. Average Sales Cycle: {avg_cycle:.2f} days")

print("\nBusiness Insights Generated Successfully!")
